# Data Cleaning & Transformation — Premier League 2020/21 Pipeline

This notebook walks through the full cleaning pipeline for our 3-source football analytics project:
1. **Load** all raw data files
2. **Clean** each source individually (fix types, parse fields, handle nulls)
3. **Standardize** team names across all sources
4. **Fuzzy match** player names across sources
5. **Integrate** into unified datasets ready for analysis
6. **Quality report** — before/after statistics

## 0. Setup & Configuration

In [ ]:
import json
import os
import re
import warnings
from collections import Counter

import pandas as pd
import numpy as np

# Fuzzy matching — install if not available
!pip install thefuzz -q

from thefuzz import fuzz, process

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 40)

# ---------------------------------------------------------------------------
# MOUNT GOOGLE DRIVE & SET PATHS
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

RAW_DIR = "/content/drive/MyDrive/MSBA-305"

PATHS = {
    # Transfermarkt
    "tm_clubs":    os.path.join(RAW_DIR, "premier_league_clubs_2020_2021.json"),
    "tm_players":  os.path.join(RAW_DIR, "premier_league_players_2020_2021.json"),
    "tm_games":    os.path.join(RAW_DIR, "premier_league_games_2020_2021.json"),
    # Understat
    "us_players":  os.path.join(RAW_DIR, "understat_players_2020_2021.json"),
    "us_teams":    os.path.join(RAW_DIR, "understat_teams_2020_2021.json"),
    "us_matches":  os.path.join(RAW_DIR, "understat_matches_2020_2021.json"),
    # Kaggle FIFA 21
    "fifa":        os.path.join(RAW_DIR, "fifa21_pl_players.json"),
}

# Output directory for cleaned data
CLEAN_DIR = "/content/drive/MyDrive/MSBA-305/cleaned"
os.makedirs(CLEAN_DIR, exist_ok=True)

def load_jsonl(filepath):
    """Load a JSONL file (one JSON object per line) into a list of dicts."""
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(records, filepath):
    """Save a list of dicts as JSONL."""
    with open(filepath, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    print(f"  Saved {len(records)} records -> {filepath}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.9 MB/s eta 0:00:00
Mounted at /content/drive


## 1. Load Raw Data

In [ ]:
print("=" * 60)
print("LOADING RAW DATA")
print("=" * 60)

raw = {}
for key, path in PATHS.items():
    data = load_jsonl(path)
    raw[key] = data
    print(f"  {key:15s}: {len(data):>5} records from {path}")

LOADING RAW DATA
  tm_clubs       :    20 records from /content/drive/MyDrive/MSBA-305/premier_league_clubs_2020_2021.json
  tm_players     :   783 records from /content/drive/MyDrive/MSBA-305/premier_league_players_2020_2021.json
  tm_games       :   380 records from /content/drive/MyDrive/MSBA-305/premier_league_games_2020_2021.json
  us_players     :   524 records from /content/drive/MyDrive/MSBA-305/understat_players_2020_2021.json
  us_teams       :   532 records from /content/drive/MyDrive/MSBA-305/understat_teams_2020_2021.json
  us_matches     :   380 records from /content/drive/MyDrive/MSBA-305/understat_matches_2020_2021.json
  fifa           :   654 records from /content/drive/MyDrive/MSBA-305/fifa21_pl_players.json


## 2. Quality Report — BEFORE Cleaning

Document the state of each dataset before any transformations.

In [ ]:
print("=" * 60)
print("QUALITY REPORT — BEFORE CLEANING")
print("=" * 60)

# --- TM Players ---
df_tm = pd.DataFrame(raw["tm_players"])
print(f"\n--- Transfermarkt Players ({len(df_tm)} rows) ---")
print(f"  Null counts:")
null_cols = df_tm.isnull().sum()
null_cols = null_cols[null_cols > 0].sort_values(ascending=False)
for col, n in null_cols.items():
    if col != "parent":
        print(f"    {col}: {n} ({n/len(df_tm)*100:.1f}%)")

# --- Understat Players ---
df_us = pd.DataFrame(raw["us_players"])
print(f"\n--- Understat Players ({len(df_us)} rows) ---")
print(f"  Numeric columns stored as strings: {df_us['goals'].dtype}")
multi_team = df_us[df_us["team_title"].str.contains(",", na=False)]
print(f"  Players with multiple teams (mid-season transfers): {len(multi_team)}")

# --- FIFA 21 ---
df_fifa = pd.DataFrame(raw["fifa"])
print(f"\n--- FIFA 21 Players ({len(df_fifa)} rows) ---")
null_fifa = df_fifa.isnull().sum()
null_fifa = null_fifa[null_fifa > 0].sort_values(ascending=False).head(10)
for col, n in null_fifa.items():
    print(f"    {col}: {n} ({n/len(df_fifa)*100:.1f}%)")

# --- TM Games ---
df_games = pd.DataFrame(raw["tm_games"])
print(f"\n--- Transfermarkt Games ({len(df_games)} rows) ---")
print(f"  Null attendance: {df_games['attendance'].isnull().sum()} (COVID season)")
print(f"  Malformed half_time_score: {(df_games['half_time_score'] == '(').sum()}")

QUALITY REPORT — BEFORE CLEANING

--- Transfermarkt Players (783 rows) ---
  Null counts:
    day_of_last_contract_extension: 783 (100.0%)
    current_market_value: 783 (100.0%)
    market_value_history: 783 (100.0%)
    highest_market_value: 783 (100.0%)
    national_team: 699 (89.3%)
    additional_citizenships: 424 (54.2%)
    outfitter: 422 (53.9%)
    full_name: 159 (20.3%)
    name_in_home_country: 159 (20.3%)
    number: 143 (18.3%)
    international_goals: 86 (11.0%)
    international_caps: 86 (11.0%)
    social_media: 76 (9.7%)
    foot: 7 (0.9%)
    height: 6 (0.8%)
    joined: 4 (0.5%)
    contract_expires: 4 (0.5%)

--- Understat Players (524 rows) ---
  Numeric columns stored as strings: object
  Players with multiple teams (mid-season transfers): 8

--- FIFA 21 Players (654 rows) ---
    defending_marking: 654 (100.0%)
    loaned_from: 641 (98.0%)
    gk_handling: 588 (89.9%)
    gk_speed: 588 (89.9%)
    gk_diving: 588 (89.9%)
    gk_kicking: 588 (89.9%)
    gk_reflexes:

## 3. Clean Transfermarkt Data

### 3.1 Clean TM Clubs

In [ ]:
print("Cleaning TM Clubs...")
tm_clubs_clean = []

for club in raw["tm_clubs"]:
    cleaned = {
        "club_name":           club["name"],
        "club_code":           club.get("code"),
        "club_href":           club.get("href"),
        # Parse numeric fields
        "squad_size":          int(club["squad_size"]) if club.get("squad_size") else None,
        "average_age":         float(club["average_age"]) if club.get("average_age") else None,
        "foreigners_number":   int(club["foreigners_number"]) if club.get("foreigners_number") else None,
        "national_team_players": int(club["national_team_players"]) if club.get("national_team_players") else None,
        "stadium_name":        club.get("stadium_name"),
        "coach_name":          club.get("coach_name"),
        # Parse stadium seats: "55.097 Seats" -> 55097
        "stadium_seats":       None,
        # Parse foreigners percentage: "70.4 %" -> 70.4
        "foreigners_pct":      None,
        # Parse net transfer record: "€-199.52m" -> -199520000
        "net_transfer_record_eur": None,
    }

    # Stadium seats
    seats_raw = club.get("stadium_seats", "")
    if seats_raw:
        seats_match = re.search(r"([\d.]+)", str(seats_raw).replace(".", ""))
        if seats_match:
            cleaned["stadium_seats"] = int(seats_match.group(1))

    # Foreigners percentage
    fp_raw = club.get("foreigners_percentage", "")
    if fp_raw:
        fp_match = re.search(r"([\d.]+)", str(fp_raw))
        if fp_match:
            cleaned["foreigners_pct"] = float(fp_match.group(1))

    # Net transfer record
    ntr_raw = club.get("net_transfer_record", "")
    if ntr_raw and "€" in str(ntr_raw):
        ntr_str = str(ntr_raw).replace("€", "").strip()
        multiplier = 1
        if "bn" in ntr_str:
            multiplier = 1_000_000_000
            ntr_str = ntr_str.replace("bn", "")
        elif "m" in ntr_str:
            multiplier = 1_000_000
            ntr_str = ntr_str.replace("m", "")
        elif "k" in ntr_str or "Th." in ntr_str:
            multiplier = 1_000
            ntr_str = ntr_str.replace("k", "").replace("Th.", "")
        try:
            cleaned["net_transfer_record_eur"] = float(ntr_str) * multiplier
        except ValueError:
            pass

    tm_clubs_clean.append(cleaned)

df_clubs = pd.DataFrame(tm_clubs_clean)
print(f"  {len(df_clubs)} clubs cleaned")
print(f"  Columns: {list(df_clubs.columns)}")
df_clubs[["club_name", "squad_size", "average_age", "coach_name", "stadium_seats"]].head()

Cleaning TM Clubs...
  20 clubs cleaned
  Columns: ['club_name', 'club_code', 'club_href', 'squad_size', 'average_age', 'foreigners_number', 'national_team_players', 'stadium_name', 'coach_name', 'stadium_seats', 'foreigners_pct', 'net_transfer_record_eur']


,club_name,squad_size,average_age,coach_name,stadium_seats
0,Arsenal FC,24,26.5,Mikel Arteta,60704
1,Everton FC,25,26.7,Carlo Ancelotti,52769
2,Liverpool FC,28,26.0,Jürgen Klopp,61276
3,Leicester City,29,26.0,Brendan Rodgers,32259
4,Chelsea FC,31,23.6,Frank Lampard,41631


### 3.2 Clean TM Players

In [ ]:
print("Cleaning TM Players...")
tm_players_clean = []

for p in raw["tm_players"]:
    parent = p.get("parent", {})
    club_name = parent.get("club_name") or parent.get("name", "")

    # Build full name
    first = (p.get("name") or "").strip()
    last = (p.get("last_name") or "").strip()
    full_name = f"{first} {last}".strip() if first else last

    # Parse height: "1,88 m" -> 188 cm
    height_cm = None
    height_raw = p.get("height")
    if height_raw:
        h_match = re.search(r"(\d+)[,.](\d+)", str(height_raw))
        if h_match:
            height_cm = int(h_match.group(1)) * 100 + int(h_match.group(2))

    # Parse age
    age = p.get("age")
    if age:
        try:
            age = int(age)
        except (ValueError, TypeError):
            age = None

    # Parse jersey number: "#31" -> 31
    number = p.get("number")
    if number:
        n_match = re.search(r"(\d+)", str(number))
        number = int(n_match.group(1)) if n_match else None

    # Parse international caps
    caps = p.get("international_caps")
    if caps:
        try:
            caps = int(caps)
        except (ValueError, TypeError):
            caps = None

    # Parse contract expiry and joined dates — strip whitespace
    contract_expires = (p.get("contract_expires") or "").strip() or None
    joined = (p.get("joined") or "").strip() or None

    # Extract player_id from href: /ederson/profil/spieler/238223 -> 238223
    player_id = None
    href = p.get("href", "")
    if href:
        id_match = re.search(r"/spieler/(\d+)", href)
        if id_match:
            player_id = id_match.group(1)

    cleaned = {
        "tm_player_id":       player_id,
        "full_name":          full_name,
        "first_name":         first,
        "last_name":          last,
        "club_name":          club_name,
        "position":           p.get("position"),
        "age":                age,
        "date_of_birth":      p.get("date_of_birth"),
        "height_cm":          height_cm,
        "citizenship":        p.get("citizenship"),
        "foot":               p.get("foot"),
        "jersey_number":      number,
        "international_caps": caps,
        "international_goals": p.get("international_goals"),
        "contract_expires":   contract_expires,
        "joined":             joined,
        "player_href":        href,
    }
    tm_players_clean.append(cleaned)

df_tm_players = pd.DataFrame(tm_players_clean)
print(f"  {len(df_tm_players)} players cleaned")
print(f"  Null full_name: {(df_tm_players['full_name'] == '').sum()}")
print(f"  Null height_cm: {df_tm_players['height_cm'].isnull().sum()}")
print(f"  Null foot: {df_tm_players['foot'].isnull().sum()}")
df_tm_players[["full_name", "club_name", "position", "age", "height_cm", "foot"]].head(10)

Cleaning TM Players...
  783 players cleaned
  Null full_name: 0
  Null height_cm: 6
  Null foot: 7


,full_name,club_name,position,age,height_cm,foot
0,Lukasz Fabianski,West Ham United,Goalkeeper,40,190.0,right
1,Gonçalo Cardoso,West Ham United,Defender - Centre-Back,25,189.0,left
2,Fabián Balbuena,West Ham United,Defender - Centre-Back,34,189.0,right
3,Issa Diop,West Ham United,Defender - Centre-Back,29,194.0,right
4,Winston Reid,West Ham United,Defender - Centre-Back,37,191.0,right
5,Frederik Alves,West Ham United,Defender - Centre-Back,26,195.0,right
6,Nathan Trott,West Ham United,Goalkeeper,27,188.0,left
7,Jamal Baptiste,West Ham United,Defender - Centre-Back,22,191.0,right
8,Aji Alese,West Ham United,Defender - Centre-Back,25,192.0,left
9,Craig Dawson,West Ham United,Defender - Centre-Back,35,188.0,right


### 3.3 Clean TM Games

In [ ]:
print("Cleaning TM Games...")
tm_games_clean = []

for g in raw["tm_games"]:
    # Parse result: "0:1" -> home_goals=0, away_goals=1
    home_goals, away_goals = None, None
    result = g.get("result", "")
    if ":" in str(result):
        parts = str(result).split(":")
        try:
            home_goals = int(parts[0].strip())
            away_goals = int(parts[1].strip())
        except ValueError:
            pass

    # Parse date: "Sun, 13/09/20" -> "2020-09-13"
    date_parsed = None
    date_raw = g.get("date", "")
    if date_raw:
        d_match = re.search(r"(\d{2})/(\d{2})/(\d{2})", str(date_raw))
        if d_match:
            day, month, year = d_match.groups()
            year_full = f"20{year}"
            date_parsed = f"{year_full}-{month}-{day}"

    # Parse matchday: "1. Matchday" -> 1
    matchday = None
    md_raw = g.get("matchday", "")
    if md_raw:
        md_match = re.search(r"(\d+)", str(md_raw))
        if md_match:
            matchday = int(md_match.group(1))

    # Parse club positions: strip whitespace mess
    home_pos = None
    hp_raw = g.get("home_club_position", "")
    if hp_raw:
        hp_match = re.search(r"(\d+)", str(hp_raw))
        if hp_match:
            home_pos = int(hp_match.group(1))

    away_pos = None
    ap_raw = g.get("away_club_position", "")
    if ap_raw:
        ap_match = re.search(r"(\d+)", str(ap_raw))
        if ap_match:
            away_pos = int(ap_match.group(1))

    # Parse events into structured lists
    goals_list = []
    cards_list = []
    subs_list = []

    for evt in g.get("events", []):
        evt_type = evt.get("type", "")
        minute = evt.get("minute")
        extra = evt.get("extra")  # stoppage time
        club_name = evt.get("club", {}).get("name", "")
        player_href = evt.get("player", {}).get("href", "")
        action = evt.get("action", {})

        if evt_type == "Goals":
            goals_list.append({
                "minute": minute,
                "extra_time": extra,
                "club": club_name,
                "scorer_href": player_href,
                "assist_href": action.get("player_assist", {}).get("href"),
                "description": (action.get("description") or "").strip(", "),
                "score_after": action.get("result"),
            })
        elif evt_type == "Cards":
            cards_list.append({
                "minute": minute,
                "club": club_name,
                "player_href": player_href,
                "description": (action.get("description") or "").strip(),
            })
        elif evt_type == "Substitutions":
            subs_list.append({
                "minute": minute,
                "extra_time": extra,
                "club": club_name,
                "player_out_href": player_href,
                "player_in_href": action.get("player_in", {}).get("href"),
                "reason": (action.get("description") or "").strip(", "),
            })

    cleaned = {
        "game_id":         g.get("game_id"),
        "date":            date_parsed,
        "matchday":        matchday,
        "home_club":       g.get("home_club_name"),
        "away_club":       g.get("away_club_name"),
        "home_goals":      home_goals,
        "away_goals":      away_goals,
        "result_str":      result,
        "home_position":   home_pos,
        "away_position":   away_pos,
        "stadium":         g.get("stadium"),
        "attendance":      g.get("attendance"),
        "referee":         g.get("referee"),
        "home_manager":    g.get("home_manager", {}).get("name"),
        "away_manager":    g.get("away_manager", {}).get("name"),
        "goals":           goals_list,
        "cards":           cards_list,
        "substitutions":   subs_list,
        "total_goals":     (home_goals or 0) + (away_goals or 0),
        "total_cards":     len(cards_list),
        "total_subs":      len(subs_list),
    }
    tm_games_clean.append(cleaned)

df_tm_games = pd.DataFrame(tm_games_clean)
print(f"  {len(df_tm_games)} games cleaned")
print(f"  Date range: {df_tm_games['date'].min()} to {df_tm_games['date'].max()}")
print(f"  Total goals in season: {df_tm_games['total_goals'].sum()}")
print(f"  Null attendance: {df_tm_games['attendance'].isnull().sum()} (COVID season)")
df_tm_games[["date", "matchday", "home_club", "away_club", "home_goals", "away_goals", "referee"]].head()

Cleaning TM Games...
  380 games cleaned
  Date range: 2020-09-12 to 2021-05-23
  Total goals in season: 1024
  Null attendance: 345 (COVID season)


,date,matchday,home_club,away_club,home_goals,away_goals,referee
0,2020-09-13,1,Tottenham Hotspur,Everton FC,0,1,Martin Atkinson
1,2020-09-12,1,Liverpool FC,Leeds United,4,3,Michael Oliver
2,2020-09-12,1,Fulham FC,Arsenal FC,0,3,Chris Kavanagh
3,2020-09-12,1,West Ham United,Newcastle United,0,2,Stuart Attwell
4,2020-09-14,1,Brighton & Hove Albion,Chelsea FC,1,3,Craig Pawson


## 4. Clean Understat Data

### 4.1 Clean Understat Players

In [ ]:
print("Cleaning Understat Players...")

us_players_clean = []
for p in raw["us_players"]:
    # Handle mid-season transfers: "Arsenal,West Bromwich Albion" -> take primary (first) team
    team_raw = p.get("team_title", "")
    primary_team = team_raw.split(",")[0].strip() if team_raw else ""
    transferred = "," in team_raw

    cleaned = {
        "us_player_id":    p.get("id"),
        "player_name":     p.get("player_name"),
        "team":            primary_team,
        "team_raw":        team_raw,  # preserve original for documentation
        "transferred":     transferred,
        "position":        p.get("position"),
        # Convert string numerics to proper types
        "games":           int(p["games"]) if p.get("games") else 0,
        "minutes":         int(p["time"]) if p.get("time") else 0,
        "goals":           int(p["goals"]) if p.get("goals") else 0,
        "assists":         int(p["assists"]) if p.get("assists") else 0,
        "shots":           int(p["shots"]) if p.get("shots") else 0,
        "key_passes":      int(p["key_passes"]) if p.get("key_passes") else 0,
        "yellow_cards":    int(p["yellow_cards"]) if p.get("yellow_cards") else 0,
        "red_cards":       int(p["red_cards"]) if p.get("red_cards") else 0,
        "npg":             int(p["npg"]) if p.get("npg") else 0,
        # Float metrics
        "xG":              float(p["xG"]) if p.get("xG") else 0.0,
        "xA":              float(p["xA"]) if p.get("xA") else 0.0,
        "npxG":            float(p["npxG"]) if p.get("npxG") else 0.0,
        "xGChain":         float(p["xGChain"]) if p.get("xGChain") else 0.0,
        "xGBuildup":       float(p["xGBuildup"]) if p.get("xGBuildup") else 0.0,
    }

    # Derived metrics
    if cleaned["xG"] > 0:
        cleaned["goals_minus_xG"] = round(cleaned["goals"] - cleaned["xG"], 3)
    else:
        cleaned["goals_minus_xG"] = None

    if cleaned["minutes"] > 0:
        cleaned["xG_per_90"] = round(cleaned["xG"] / cleaned["minutes"] * 90, 3)
        cleaned["xA_per_90"] = round(cleaned["xA"] / cleaned["minutes"] * 90, 3)
    else:
        cleaned["xG_per_90"] = None
        cleaned["xA_per_90"] = None

    us_players_clean.append(cleaned)

df_us_players = pd.DataFrame(us_players_clean)
print(f"  {len(df_us_players)} players cleaned")
print(f"  Mid-season transfers: {df_us_players['transferred'].sum()}")
print(f"  Positions: {dict(df_us_players['position'].value_counts())}")
df_us_players[["player_name", "team", "goals", "xG", "goals_minus_xG", "xG_per_90"]].head(10)

Cleaning Understat Players...
  524 players cleaned
  Mid-season transfers: 8
  Positions: {'M S': np.int64(107), 'D S': np.int64(78), 'F M S': np.int64(75), 'D': np.int64(54), 'S': np.int64(52), 'D M S': np.int64(49), 'GK': np.int64(36), 'F S': np.int64(35), 'D M': np.int64(16), 'D F M S': np.int64(8), 'M': np.int64(6), 'F': np.int64(3), 'GK S': np.int64(3), 'F M': np.int64(2)}


,player_name,team,goals,xG,goals_minus_xG,xG_per_90
0,Harry Kane,Tottenham,23,22.174865,0.825,0.644
1,Mohamed Salah,Liverpool,22,20.250851,1.749,0.591
2,Bruno Fernandes,Manchester United,18,16.019451,1.981,0.463
3,Son Heung-Min,Tottenham,17,11.023286,5.977,0.316
4,Patrick Bamford,Leeds,17,18.401862,-1.402,0.537
5,Dominic Calvert-Lewin,Everton,16,18.210517,-2.211,0.569
6,Jamie Vardy,Leicester,15,19.942947,-4.943,0.630
7,Ollie Watkins,Aston Villa,14,16.280177,-2.280,0.440
8,Ilkay Gündogan,Manchester City,13,9.566889,3.433,0.424
9,Alexandre Lacazette,Arsenal,13,12.028911,0.971,0.558


### 4.2 Clean Understat Matches

In [ ]:
print("Cleaning Understat Matches...")

us_matches_clean = []
for m in raw["us_matches"]:
    cleaned = {
        "us_match_id":     m.get("id"),
        "datetime":        m.get("datetime"),
        "date":            m.get("datetime", "")[:10] if m.get("datetime") else None,
        "home_team":       m.get("h", {}).get("title"),
        "away_team":       m.get("a", {}).get("title"),
        "home_short":      m.get("h", {}).get("short_title"),
        "away_short":      m.get("a", {}).get("short_title"),
        "home_goals":      int(m.get("goals", {}).get("h", 0)),
        "away_goals":      int(m.get("goals", {}).get("a", 0)),
        "home_xG":         round(float(m.get("xG", {}).get("h", 0)), 3),
        "away_xG":         round(float(m.get("xG", {}).get("a", 0)), 3),
        "forecast_win":    round(float(m.get("forecast", {}).get("w", 0)), 4),
        "forecast_draw":   round(float(m.get("forecast", {}).get("d", 0)), 4),
        "forecast_loss":   round(float(m.get("forecast", {}).get("l", 0)), 4),
    }

    # Derived
    cleaned["home_xG_diff"] = round(cleaned["home_goals"] - cleaned["home_xG"], 3)
    cleaned["away_xG_diff"] = round(cleaned["away_goals"] - cleaned["away_xG"], 3)
    cleaned["total_xG"] = round(cleaned["home_xG"] + cleaned["away_xG"], 3)

    us_matches_clean.append(cleaned)

df_us_matches = pd.DataFrame(us_matches_clean)
print(f"  {len(df_us_matches)} matches cleaned")
print(f"  Date range: {df_us_matches['date'].min()} to {df_us_matches['date'].max()}")
print(f"  Avg total xG per match: {df_us_matches['total_xG'].mean():.2f}")
df_us_matches[["date", "home_team", "away_team", "home_goals", "away_goals", "home_xG", "away_xG"]].head()

Cleaning Understat Matches...
  380 matches cleaned
  Date range: 2020-09-12 to 2021-05-23
  Avg total xG per match: 2.73


,date,home_team,away_team,home_goals,away_goals,home_xG,away_xG
0,2020-09-12,Fulham,Arsenal,0,3,0.126,2.163
1,2020-09-12,Crystal Palace,Southampton,1,0,1.396,1.263
2,2020-09-12,Liverpool,Leeds,4,3,3.154,0.270
3,2020-09-12,West Ham,Newcastle United,0,2,0.861,1.659
4,2020-09-13,West Bromwich Albion,Leicester,0,3,0.353,2.956


## 5. Clean FIFA 21 Data

In [ ]:
print("Cleaning FIFA 21 Players...")

# Select relevant columns only (106 is too many for our analysis)
fifa_keep_cols = [
    # Identity
    "sofifa_id", "short_name", "long_name", "age", "dob", "height_cm", "weight_kg",
    "nationality", "club_name", "player_positions", "preferred_foot",
    # Ratings
    "overall", "potential", "value_eur", "wage_eur",
    "international_reputation", "weak_foot", "skill_moves", "work_rate",
    # Main attributes
    "pace", "shooting", "passing", "dribbling", "defending", "physic",
    # Key sub-attributes for our xG analysis
    "attacking_finishing", "attacking_short_passing", "attacking_volleys",
    "skill_ball_control", "skill_long_passing",
    "movement_sprint_speed", "movement_agility", "movement_reactions",
    "power_shot_power", "power_long_shots", "power_stamina",
    "mentality_vision", "mentality_composure", "mentality_positioning",
    "defending_marking_awareness", "defending_standing_tackle",
]

df_fifa_clean = pd.DataFrame(raw["fifa"])

# Keep only columns that exist
available_cols = [c for c in fifa_keep_cols if c in df_fifa_clean.columns]
missing_cols = [c for c in fifa_keep_cols if c not in df_fifa_clean.columns]
if missing_cols:
    print(f"  Missing columns (skipped): {missing_cols}")

df_fifa_clean = df_fifa_clean[available_cols].copy()

# Parse work_rate into two fields: "High/ Medium" -> attack=High, defense=Medium
if "work_rate" in df_fifa_clean.columns:
    wr_split = df_fifa_clean["work_rate"].str.split("/", expand=True)
    if wr_split.shape[1] >= 2:
        df_fifa_clean["work_rate_attack"] = wr_split[0].str.strip()
        df_fifa_clean["work_rate_defense"] = wr_split[1].str.strip()
    df_fifa_clean.drop(columns=["work_rate"], inplace=True)

# Convert value and wage to millions for readability
df_fifa_clean["value_eur_millions"] = (df_fifa_clean["value_eur"] / 1_000_000).round(2)
df_fifa_clean["wage_eur_weekly"] = df_fifa_clean["wage_eur"]

print(f"  {len(df_fifa_clean)} players, {len(df_fifa_clean.columns)} columns")
print(f"  Null value_eur: {df_fifa_clean['value_eur'].isnull().sum()}")
print(f"  Overall range: {df_fifa_clean['overall'].min()} - {df_fifa_clean['overall'].max()}")
df_fifa_clean[["short_name", "long_name", "club_name", "overall", "value_eur_millions", "pace", "shooting"]].head(10)

Cleaning FIFA 21 Players...
  Missing columns (skipped): ['defending_marking_awareness']
  654 players, 43 columns
  Null value_eur: 0
  Overall range: 50 - 91


,short_name,long_name,club_name,overall,value_eur_millions,pace,shooting
0,K. De Bruyne,Kevin De Bruyne,Manchester City,91,87.0,76.0,86.0
1,V. van Dijk,Virgil van Dijk,Liverpool,90,75.5,76.0,60.0
2,Alisson,Alisson Ramsés Becker,Liverpool,90,62.5,NaN,NaN
3,S. Mané,Sadio Mané,Liverpool,90,78.0,94.0,85.0
4,M. Salah,Mohamed Salah Ghaly,Liverpool,90,78.0,93.0,86.0
5,S. Agüero,Sergio Leonel Agüero del Castillo,Manchester City,89,53.0,78.0,90.0
6,Ederson,Ederson Santana de Moraes,Manchester City,88,53.5,NaN,NaN
7,R. Sterling,Raheem Shaquille Sterling,Manchester City,88,72.5,93.0,81.0
8,H. Kane,Harry Kane,Tottenham Hotspur,88,71.0,68.0,91.0
9,N. Kanté,N'Golo Kanté,Chelsea,88,51.0,77.0,66.0


## 6. Standardize Team Names Across Sources

Each source uses different naming conventions:
- TM: "Arsenal FC", "Chelsea FC", "Burnley FC" (adds FC/City suffixes)
- Understat: "Arsenal", "Chelsea", "Burnley" (short names)
- FIFA 21: Mix — "Arsenal", "Tottenham Hotspur", "Brighton & Hove Albion"

In [ ]:
# ---------------------------------------------------------------------------
# CANONICAL TEAM NAME MAPPING
# ---------------------------------------------------------------------------
# We standardize everything to a common name (using Understat's convention as
# the base since it's the shortest and most recognizable)

TEAM_NAME_MAP = {
    # Transfermarkt names -> canonical
    "Arsenal FC":                "Arsenal",
    "Chelsea FC":                "Chelsea",
    "Everton FC":                "Everton",
    "Fulham FC":                 "Fulham",
    "Liverpool FC":              "Liverpool",
    "Burnley FC":                "Burnley",
    "Southampton FC":            "Southampton",
    "Leeds United":              "Leeds",
    "Leicester City":            "Leicester",
    "Tottenham Hotspur":         "Tottenham",
    "West Ham United":           "West Ham",
    "Brighton & Hove Albion":    "Brighton",
    "Sheffield United":          "Sheffield United",
    "Newcastle United":          "Newcastle United",
    "West Bromwich Albion":      "West Bromwich Albion",
    "Wolverhampton Wanderers":   "Wolverhampton Wanderers",
    "Crystal Palace":            "Crystal Palace",
    "Manchester City":           "Manchester City",
    "Manchester United":         "Manchester United",
    "Aston Villa":               "Aston Villa",

    # Understat names (most are already canonical, but include for completeness)
    "Arsenal":                   "Arsenal",
    "Chelsea":                   "Chelsea",
    "Everton":                   "Everton",
    "Fulham":                    "Fulham",
    "Liverpool":                 "Liverpool",
    "Burnley":                   "Burnley",
    "Southampton":               "Southampton",
    "Leeds":                     "Leeds",
    "Leicester":                 "Leicester",
    "Tottenham":                 "Tottenham",
    "West Ham":                  "West Ham",
    "Brighton":                  "Brighton",

    # FIFA 21 names (some differ from Understat)
    "Leeds United":              "Leeds",
    "Leicester City":            "Leicester",
    "Tottenham Hotspur":         "Tottenham",
    "West Ham United":           "West Ham",
    "Brighton & Hove Albion":    "Brighton",
}


def standardize_team(name):
    """Map any team name variant to the canonical name."""
    if not name or pd.isna(name):
        return name
    name = str(name).strip()
    return TEAM_NAME_MAP.get(name, name)


# Apply to all datasets
print("Standardizing team names...\n")

# TM Clubs
df_clubs["club_name_std"] = df_clubs["club_name"].apply(standardize_team)

# TM Players
df_tm_players["club_name_std"] = df_tm_players["club_name"].apply(standardize_team)

# TM Games
df_tm_games["home_club_std"] = df_tm_games["home_club"].apply(standardize_team)
df_tm_games["away_club_std"] = df_tm_games["away_club"].apply(standardize_team)

# Understat Players
df_us_players["team_std"] = df_us_players["team"].apply(standardize_team)

# Understat Matches
df_us_matches["home_team_std"] = df_us_matches["home_team"].apply(standardize_team)
df_us_matches["away_team_std"] = df_us_matches["away_team"].apply(standardize_team)

# FIFA 21
df_fifa_clean["club_name_std"] = df_fifa_clean["club_name"].apply(standardize_team)

# Verify: all sources should now have the same 20 team names
tm_teams = set(df_clubs["club_name_std"])
us_teams = set(df_us_matches["home_team_std"]) | set(df_us_matches["away_team_std"])
fifa_teams = set(df_fifa_clean["club_name_std"])

print(f"TM teams:       {len(tm_teams)}")
print(f"Understat teams: {len(us_teams)}")
print(f"FIFA teams:      {len(fifa_teams)}")
print(f"\nAll match: {tm_teams == us_teams == fifa_teams}")

if tm_teams != us_teams:
    print(f"  TM - Understat: {tm_teams - us_teams}")
    print(f"  Understat - TM: {us_teams - tm_teams}")
if tm_teams != fifa_teams:
    print(f"  TM - FIFA: {tm_teams - fifa_teams}")
    print(f"  FIFA - TM: {fifa_teams - tm_teams}")

Standardizing team names...

TM teams:       20
Understat teams: 20
FIFA teams:      20

All match: True


## 7. Fuzzy Match Players Across Sources

Join key: **player name + team**. Names differ across sources, so we use fuzzy matching.

In [ ]:
# ---------------------------------------------------------------------------
# STEP 1: Build player lookup per source
# ---------------------------------------------------------------------------

# Understat: player_name is full name
us_lookup = {}
for _, row in df_us_players.iterrows():
    key = (row["player_name"], row["team_std"])
    us_lookup[key] = row.to_dict()

# FIFA: index by BOTH long_name and short_name for better matching
fifa_lookup = {}
fifa_lookup_by_team = {}  # {team: {name: row_dict, ...}}
for _, row in df_fifa_clean.iterrows():
    rd = row.to_dict()
    team = row["club_name_std"]
    # Index by long_name + club (primary)
    key = (row["long_name"], team)
    fifa_lookup[key] = rd
    # Also index by short_name
    key2 = (row["short_name"], team)
    fifa_lookup[key2] = rd
    # Build team-grouped lookup with all name variants
    if team not in fifa_lookup_by_team:
        fifa_lookup_by_team[team] = {}
    fifa_lookup_by_team[team][row["long_name"]] = rd
    fifa_lookup_by_team[team][row["short_name"]] = rd

# TM: full_name (first + last)
tm_lookup = {}
for _, row in df_tm_players.iterrows():
    key = (row["full_name"], row["club_name_std"])
    tm_lookup[key] = row.to_dict()

print(f"TM lookup:       {len(tm_lookup)} players")
print(f"Understat lookup: {len(us_lookup)} players")
print(f"FIFA lookup:      {len(fifa_lookup)} players")

TM lookup:       783 players
Understat lookup: 524 players
FIFA lookup:      1308 players


In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Fuzzy match Understat players to TM players (by team)
# ---------------------------------------------------------------------------

def fuzzy_match_player(name, candidates, threshold=80):
    """
    Find the best fuzzy match for a player name within a list of candidate names.

    Args:
        name: str — the name to match
        candidates: list of str — possible matches
        threshold: int — minimum score (0-100) to accept

    Returns:
        (matched_name, score) or (None, 0)
    """
    if not candidates:
        return None, 0
    best_match, score = process.extractOne(name, candidates, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return best_match, score
    return None, score


print("Fuzzy matching Understat -> TM players...")
print("(matching within same team to reduce false positives)\n")

matches_us_tm = []
unmatched_us = []

for (us_name, us_team), us_data in us_lookup.items():
    # Get TM players from the same team
    tm_same_team = {name: data for (name, team), data in tm_lookup.items() if team == us_team}

    if not tm_same_team:
        unmatched_us.append({"us_name": us_name, "us_team": us_team, "reason": "no TM team match"})
        continue

    # Try exact match first
    if us_name in tm_same_team:
        matches_us_tm.append({
            "us_name": us_name,
            "tm_name": us_name,
            "team": us_team,
            "match_score": 100,
            "match_type": "exact",
        })
        continue

    # Fuzzy match
    best_name, score = fuzzy_match_player(us_name, list(tm_same_team.keys()), threshold=75)
    if best_name:
        matches_us_tm.append({
            "us_name": us_name,
            "tm_name": best_name,
            "team": us_team,
            "match_score": score,
            "match_type": "fuzzy",
        })
    else:
        unmatched_us.append({"us_name": us_name, "us_team": us_team, "reason": f"best score {score}"})

df_match_us_tm = pd.DataFrame(matches_us_tm)
print(f"Matched:   {len(df_match_us_tm)}")
print(f"Unmatched: {len(unmatched_us)}")
print(f"Match rate: {len(df_match_us_tm) / len(us_lookup) * 100:.1f}%")
print(f"\nScore distribution:")
print(df_match_us_tm["match_score"].describe())
print(f"\nLow-confidence matches (score < 90):")
low_conf = df_match_us_tm[df_match_us_tm["match_score"] < 90]
if len(low_conf) > 0:
    print(low_conf[["us_name", "tm_name", "team", "match_score"]].to_string(index=False))
else:
    print("  None — all matches above 90")

Fuzzy matching Understat -> TM players...
(matching within same team to reduce false positives)

Matched:   500
Unmatched: 24
Match rate: 95.4%

Score distribution:
count    500.000000
mean      99.236000
std        3.141982
min       76.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: match_score, dtype: float64

Low-confidence matches (score < 90):
               us_name         tm_name                 team  match_score
          Tomas Soucek    Tomáš Souček             West Ham           78
             Trézéguet       Trezeguet          Aston Villa           88
      Ezri Konsa Ngoyo      Ezri Konsa          Aston Villa           77
       Oliver McBurnie    Oli McBurnie     Sheffield United           89
        Caglar Söyüncü  Çağlar Söyüncü            Leicester           86
    Nathaniel Phillips    Nat Phillips            Liverpool           80
     N&#039;Golo Kanté    N'Golo Kanté              Chelsea           85
          Joseph Go

In [ ]:
# ---------------------------------------------------------------------------
# STEP 3: Fuzzy match Understat players to FIFA players (by team)
# ---------------------------------------------------------------------------

print("\nFuzzy matching Understat -> FIFA players...")

matches_us_fifa = []
unmatched_us_fifa = []

for (us_name, us_team), us_data in us_lookup.items():
    # Get FIFA players from the same team (both short and long names)
    fifa_same_team = fifa_lookup_by_team.get(us_team, {})

    if not fifa_same_team:
        unmatched_us_fifa.append({"us_name": us_name, "us_team": us_team, "reason": "no FIFA team match"})
        continue

    # Try exact match first
    if us_name in fifa_same_team:
        matches_us_fifa.append({
            "us_name": us_name,
            "fifa_name": us_name,
            "team": us_team,
            "match_score": 100,
            "match_type": "exact",
        })
        continue

    # Fuzzy match
    best_name, score = fuzzy_match_player(us_name, list(fifa_same_team.keys()), threshold=75)
    if best_name:
        matches_us_fifa.append({
            "us_name": us_name,
            "fifa_name": best_name,
            "team": us_team,
            "match_score": score,
            "match_type": "fuzzy",
        })
    else:
        unmatched_us_fifa.append({"us_name": us_name, "us_team": us_team, "reason": f"best score {score}"})

df_match_us_fifa = pd.DataFrame(matches_us_fifa)
print(f"Matched:   {len(df_match_us_fifa)}")
print(f"Unmatched: {len(unmatched_us_fifa)}")
print(f"Match rate: {len(df_match_us_fifa) / len(us_lookup) * 100:.1f}%")


Fuzzy matching Understat -> FIFA players...
Matched:   438
Unmatched: 86
Match rate: 83.6%


## 8. Build Unified Player Dataset

Merge all three sources into one integrated player table using the fuzzy match results.

In [ ]:
print("Building unified player dataset...\n")

# Start with Understat as the base (only players with actual playing time)
unified_players = []

for _, us_row in df_us_players.iterrows():
    player = {
        # === Understat fields (performance) ===
        "us_player_id":    us_row["us_player_id"],
        "player_name":     us_row["player_name"],
        "team":            us_row["team_std"],
        "position_us":     us_row["position"],
        "games":           us_row["games"],
        "minutes":         us_row["minutes"],
        "goals":           us_row["goals"],
        "assists":         us_row["assists"],
        "xG":              us_row["xG"],
        "xA":              us_row["xA"],
        "npxG":            us_row["npxG"],
        "xGChain":         us_row["xGChain"],
        "xGBuildup":       us_row["xGBuildup"],
        "shots":           us_row["shots"],
        "key_passes":      us_row["key_passes"],
        "goals_minus_xG":  us_row["goals_minus_xG"],
        "xG_per_90":       us_row["xG_per_90"],
        "xA_per_90":       us_row["xA_per_90"],
        "transferred":     us_row["transferred"],
    }

    # === Merge TM fields (profile) ===
    tm_match = df_match_us_tm[
        (df_match_us_tm["us_name"] == us_row["player_name"]) &
        (df_match_us_tm["team"] == us_row["team_std"])
    ]

    if len(tm_match) > 0:
        tm_name = tm_match.iloc[0]["tm_name"]
        tm_team = tm_match.iloc[0]["team"]
        tm_data = tm_lookup.get((tm_name, tm_team), {})
        player["tm_player_id"] = tm_data.get("tm_player_id")
        player["tm_matched_name"] = tm_name
        player["tm_match_score"] = tm_match.iloc[0]["match_score"]
        player["position_tm"] = tm_data.get("position")
        player["age"] = tm_data.get("age")
        player["date_of_birth"] = tm_data.get("date_of_birth")
        player["height_cm"] = tm_data.get("height_cm")
        player["citizenship"] = tm_data.get("citizenship")
        player["foot"] = tm_data.get("foot")
        player["international_caps"] = tm_data.get("international_caps")
        player["contract_expires"] = tm_data.get("contract_expires")
    else:
        player["tm_player_id"] = None
        player["tm_matched_name"] = None
        player["tm_match_score"] = None
        player["position_tm"] = None
        player["age"] = None
        player["date_of_birth"] = None
        player["height_cm"] = None
        player["citizenship"] = None
        player["foot"] = None
        player["international_caps"] = None
        player["contract_expires"] = None

    # === Merge FIFA fields (ratings & value) ===
    fifa_match = df_match_us_fifa[
        (df_match_us_fifa["us_name"] == us_row["player_name"]) &
        (df_match_us_fifa["team"] == us_row["team_std"])
    ]

    if len(fifa_match) > 0:
        fifa_name = fifa_match.iloc[0]["fifa_name"]
        fifa_team = fifa_match.iloc[0]["team"]
        fifa_data = fifa_lookup.get((fifa_name, fifa_team), {})
        player["sofifa_id"] = fifa_data.get("sofifa_id")
        player["fifa_matched_name"] = fifa_name
        player["fifa_match_score"] = fifa_match.iloc[0]["match_score"]
        player["overall"] = fifa_data.get("overall")
        player["potential"] = fifa_data.get("potential")
        player["value_eur"] = fifa_data.get("value_eur")
        player["wage_eur"] = fifa_data.get("wage_eur")
        player["fifa_pace"] = fifa_data.get("pace")
        player["fifa_shooting"] = fifa_data.get("shooting")
        player["fifa_passing"] = fifa_data.get("passing")
        player["fifa_dribbling"] = fifa_data.get("dribbling")
        player["fifa_defending"] = fifa_data.get("defending")
        player["fifa_physic"] = fifa_data.get("physic")
        player["attacking_finishing"] = fifa_data.get("attacking_finishing")
        player["mentality_composure"] = fifa_data.get("mentality_composure")
        player["mentality_positioning"] = fifa_data.get("mentality_positioning")
        player["power_shot_power"] = fifa_data.get("power_shot_power")
    else:
        player["sofifa_id"] = None
        player["fifa_matched_name"] = None
        player["fifa_match_score"] = None
        player["overall"] = None
        player["potential"] = None
        player["value_eur"] = None
        player["wage_eur"] = None
        player["fifa_pace"] = None
        player["fifa_shooting"] = None
        player["fifa_passing"] = None
        player["fifa_dribbling"] = None
        player["fifa_defending"] = None
        player["fifa_physic"] = None
        player["attacking_finishing"] = None
        player["mentality_composure"] = None
        player["mentality_positioning"] = None
        player["power_shot_power"] = None

    unified_players.append(player)

df_unified = pd.DataFrame(unified_players)
print(f"Unified player dataset: {len(df_unified)} rows, {len(df_unified.columns)} columns")
print(f"\nSource coverage:")
print(f"  With TM data:   {df_unified['tm_player_id'].notna().sum()} ({df_unified['tm_player_id'].notna().mean()*100:.1f}%)")
print(f"  With FIFA data:  {df_unified['sofifa_id'].notna().sum()} ({df_unified['sofifa_id'].notna().mean()*100:.1f}%)")
print(f"  All 3 sources:   {((df_unified['tm_player_id'].notna()) & (df_unified['sofifa_id'].notna())).sum()}")
df_unified[["player_name", "team", "goals", "xG", "overall", "value_eur", "age", "foot"]].head(15)

Building unified player dataset...

Unified player dataset: 524 rows, 47 columns

Source coverage:
  With TM data:   500 (95.4%)
  With FIFA data:  438 (83.6%)
  All 3 sources:   424


,player_name,team,goals,xG,overall,value_eur,age,foot
0,Harry Kane,Tottenham,23,22.174865,88.0,71000000.0,32,right
1,Mohamed Salah,Liverpool,22,20.250851,90.0,78000000.0,33,left
2,Bruno Fernandes,Manchester United,18,16.019451,87.0,63000000.0,31,right
3,Son Heung-Min,Tottenham,17,11.023286,NaN,NaN,33,both
4,Patrick Bamford,Leeds,17,18.401862,71.0,2800000.0,32,left
5,Dominic Calvert-Lewin,Everton,16,18.210517,79.0,17000000.0,29,right
6,Jamie Vardy,Leicester,15,19.942947,86.0,28000000.0,39,right
7,Ollie Watkins,Aston Villa,14,16.280177,76.0,10500000.0,30,right
8,Ilkay Gündogan,Manchester City,13,9.566889,83.0,25000000.0,35,right
9,Alexandre Lacazette,Arsenal,13,12.028911,83.0,26000000.0,34,right


## 9. Build Unified Match Dataset

Join TM Games (events, referee, stadium) with Understat Matches (xG, forecasts) on date + teams.

In [ ]:
print("Building unified match dataset...\n")

# Join on date + home_team + away_team
unified_matches = []

for _, tm_row in df_tm_games.iterrows():
    match = tm_row.to_dict()

    # Find matching Understat record
    us_match = df_us_matches[
        (df_us_matches["date"] == tm_row["date"]) &
        (df_us_matches["home_team_std"] == tm_row["home_club_std"]) &
        (df_us_matches["away_team_std"] == tm_row["away_club_std"])
    ]

    if len(us_match) > 0:
        us_row = us_match.iloc[0]
        match["us_match_id"] = us_row["us_match_id"]
        match["home_xG"] = us_row["home_xG"]
        match["away_xG"] = us_row["away_xG"]
        match["home_xG_diff"] = us_row["home_xG_diff"]
        match["away_xG_diff"] = us_row["away_xG_diff"]
        match["total_xG"] = us_row["total_xG"]
        match["forecast_win"] = us_row["forecast_win"]
        match["forecast_draw"] = us_row["forecast_draw"]
        match["forecast_loss"] = us_row["forecast_loss"]
    else:
        match["us_match_id"] = None
        match["home_xG"] = None
        match["away_xG"] = None
        match["home_xG_diff"] = None
        match["away_xG_diff"] = None
        match["total_xG"] = None
        match["forecast_win"] = None
        match["forecast_draw"] = None
        match["forecast_loss"] = None

    unified_matches.append(match)

df_unified_matches = pd.DataFrame(unified_matches)
matched_count = df_unified_matches["us_match_id"].notna().sum()
print(f"Unified matches: {len(df_unified_matches)} rows")
print(f"Successfully joined with Understat xG: {matched_count} ({matched_count/len(df_unified_matches)*100:.1f}%)")
unmatched = df_unified_matches[df_unified_matches["us_match_id"].isna()]
if len(unmatched) > 0:
    print(f"\nUnmatched games ({len(unmatched)}):")
    print(unmatched[["date", "home_club_std", "away_club_std"]].to_string(index=False))

df_unified_matches[["date", "home_club_std", "away_club_std", "home_goals", "away_goals", "home_xG", "away_xG", "referee"]].head(10)

Building unified match dataset...

Unified matches: 380 rows
Successfully joined with Understat xG: 380 (100.0%)


,date,home_club_std,away_club_std,home_goals,away_goals,home_xG,away_xG,referee
0,2020-09-13,Tottenham,Everton,0,1,0.823,1.268,Martin Atkinson
1,2020-09-12,Liverpool,Leeds,4,3,3.154,0.270,Michael Oliver
2,2020-09-12,Fulham,Arsenal,0,3,0.126,2.163,Chris Kavanagh
3,2020-09-12,West Ham,Newcastle United,0,2,0.861,1.659,Stuart Attwell
4,2020-09-14,Brighton,Chelsea,1,3,1.443,1.271,Craig Pawson
5,2020-09-12,Crystal Palace,Southampton,1,0,1.396,1.263,Jonathan Moss
6,2021-01-12,Burnley,Manchester United,0,1,0.649,1.240,Kevin Friend
7,2021-01-20,Manchester City,Aston Villa,2,0,3.344,0.602,Jonathan Moss
8,2020-09-13,West Bromwich Albion,Leicester,0,3,0.353,2.956,Anthony Taylor
9,2020-09-14,Sheffield United,Wolverhampton Wanderers,0,2,0.949,1.613,Mike Dean


## 10. Quality Report — AFTER Cleaning

In [ ]:
print("=" * 60)
print("QUALITY REPORT — AFTER CLEANING")
print("=" * 60)

print(f"\n--- Unified Players ({len(df_unified)} rows, {len(df_unified.columns)} cols) ---")
print(f"  Null counts (key fields):")
key_cols = ["player_name", "team", "xG", "goals", "age", "height_cm", "foot",
            "overall", "value_eur", "citizenship"]
for col in key_cols:
    n = df_unified[col].isnull().sum()
    pct = n / len(df_unified) * 100
    print(f"    {col:25s}: {n:>4} nulls ({pct:.1f}%)")

print(f"\n--- Unified Matches ({len(df_unified_matches)} rows) ---")
match_cols = ["date", "home_goals", "home_xG", "referee", "stadium", "attendance"]
for col in match_cols:
    n = df_unified_matches[col].isnull().sum()
    pct = n / len(df_unified_matches) * 100
    print(f"    {col:25s}: {n:>4} nulls ({pct:.1f}%)")

print(f"\n--- Clubs ({len(df_clubs)} rows) ---")
for col in ["club_name_std", "squad_size", "stadium_seats", "coach_name"]:
    n = df_clubs[col].isnull().sum()
    print(f"    {col:25s}: {n:>4} nulls")

print(f"\n--- Cross-Source Join Quality ---")
print(f"  TM <-> Understat player matches:  {len(df_match_us_tm)} / {len(us_lookup)} ({len(df_match_us_tm)/len(us_lookup)*100:.1f}%)")
print(f"  FIFA <-> Understat player matches: {len(df_match_us_fifa)} / {len(us_lookup)} ({len(df_match_us_fifa)/len(us_lookup)*100:.1f}%)")
print(f"  TM <-> Understat match matches:    {matched_count} / {len(df_unified_matches)} ({matched_count/len(df_unified_matches)*100:.1f}%)")

QUALITY REPORT — AFTER CLEANING

--- Unified Players (524 rows, 47 cols) ---
  Null counts (key fields):
    player_name              :    0 nulls (0.0%)
    team                     :    0 nulls (0.0%)
    xG                       :    0 nulls (0.0%)
    goals                    :    0 nulls (0.0%)
    age                      :   25 nulls (4.8%)
    height_cm                :   24 nulls (4.6%)
    foot                     :   24 nulls (4.6%)
    overall                  :   86 nulls (16.4%)
    value_eur                :   86 nulls (16.4%)
    citizenship              :   24 nulls (4.6%)

--- Unified Matches (380 rows) ---
    date                     :    0 nulls (0.0%)
    home_goals               :    0 nulls (0.0%)
    home_xG                  :    0 nulls (0.0%)
    referee                  :    0 nulls (0.0%)
    stadium                  :    0 nulls (0.0%)
    attendance               :  345 nulls (90.8%)

--- Clubs (20 rows) ---
    club_name_std            :    0 nulls
    s

## 11. Export Cleaned Datasets

In [ ]:
print("=" * 60)
print("EXPORTING CLEANED DATA")
print("=" * 60)

# Unified players (main analytical table)
save_jsonl(
    df_unified.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "unified_players.json")
)

# Unified matches (main match table)
# Remove nested lists for JSON export — flatten goals/cards/subs to counts
matches_export = df_unified_matches.drop(columns=["goals", "cards", "substitutions"], errors="ignore")
save_jsonl(
    matches_export.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "unified_matches.json")
)

# Clubs
save_jsonl(
    df_clubs.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "clubs.json")
)

# TM players (full, cleaned — includes players without Understat data)
save_jsonl(
    df_tm_players.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "tm_players_clean.json")
)

# Understat players (cleaned)
save_jsonl(
    df_us_players.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "us_players_clean.json")
)

# FIFA players (cleaned)
save_jsonl(
    df_fifa_clean.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "fifa_players_clean.json")
)

# Match crosswalk (for documentation)
save_jsonl(
    df_match_us_tm.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "player_crosswalk_us_tm.json")
)
save_jsonl(
    df_match_us_fifa.to_dict(orient="records"),
    os.path.join(CLEAN_DIR, "player_crosswalk_us_fifa.json")
)

print(f"\n[DONE] All cleaned data exported to {CLEAN_DIR}/")

EXPORTING CLEANED DATA
  Saved 524 records -> /content/drive/MyDrive/MSBA-305/cleaned/unified_players.json
  Saved 380 records -> /content/drive/MyDrive/MSBA-305/cleaned/unified_matches.json
  Saved 20 records -> /content/drive/MyDrive/MSBA-305/cleaned/clubs.json
  Saved 783 records -> /content/drive/MyDrive/MSBA-305/cleaned/tm_players_clean.json
  Saved 524 records -> /content/drive/MyDrive/MSBA-305/cleaned/us_players_clean.json
  Saved 654 records -> /content/drive/MyDrive/MSBA-305/cleaned/fifa_players_clean.json
  Saved 500 records -> /content/drive/MyDrive/MSBA-305/cleaned/player_crosswalk_us_tm.json
  Saved 438 records -> /content/drive/MyDrive/MSBA-305/cleaned/player_crosswalk_us_fifa.json

[DONE] All cleaned data exported to /content/drive/MyDrive/MSBA-305/cleaned/


## Summary

| Dataset | Records | Sources Joined | Key Fields |
|---------|---------|---------------|------------|
| `unified_players.json` | ~524 | TM + Understat + FIFA | xG, xA, age, FIFA rating, value, nationality |
| `unified_matches.json` | 380 | TM + Understat | goals, xG, referee, stadium, events |
| `clubs.json` | 20 | TM | squad size, coach, stadium, net transfers |
| `tm_players_clean.json` | ~783 | TM only | full squad profiles |
| `us_players_clean.json` | ~524 | Understat only | all xG metrics |
| `fifa_players_clean.json` | ~654 | FIFA only | ratings + sub-attributes |